In [ ]:
from evalplus.data import get_human_eval_plus, get_mbpp_plus
import requests, json, os
import pynvml, time, threading

path = "../data/processed/qwen_15b/pass@1/"
humaneval_filename = "humaneval_solutions.jsonl"
mbpp_filename = "mbpp_solutions.jsonl"
ollama_url = "http://host.docker.internal:11434/api/generate"

In [14]:
monitoring = False
energy = 0.0
peak_vram = 0.0

In [15]:
def monitor_GPU(handle):
	global energy, peak_vram, monitoring
	power_supported = True
	last_time = time.time()
	while monitoring:
		now = time.time()
		time_diff = now-last_time
		last_time = now

		if power_supported:
			try:
				watt = pynvml.nvmlDeviceGetPowerUsage(handle) / 1000.0 # function returns mW
				energy += watt * time_diff
			except pynvml.NVMLError_NotSupported:
				power_supported = False

		memory = pynvml.nvmlDeviceGetMemoryInfo(handle)
		vram = memory.used / (1024 ** 2)
		peak_vram = vram if vram > peak_vram else peak_vram

		time.sleep(0.1)

In [ ]:
def generate_solutions(problems, output_file):
    global energy, peak_vram, monitoring

    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0) # takes the first GPU
    energy = 0.0
    peak_vram = 0.0
    monitoring = True

    if not os.path.exists(path):
        os.makedirs(path)

    completed_tasks = set()
    if os.path.exists(output_file):
        with open(output_file, "r") as f:
            for line in f:
                data = json.loads(line.strip())
                completed_tasks.add(data["task_id"])

    if(len(problems) - len(completed_tasks)) == 0:
        print("problems solutions already generated.\n")
        return
    
    with open(output_file, "a") as f:
        thread = threading.Thread(target=monitor_GPU, args=(handle,))
        thread.start()
        start_time = time.time()

        for task_id in problems.keys():
            if(task_id in completed_tasks): continue
            problem = problems[task_id]
            prompt = problem["prompt"]

            payload = {
                "model": "qwen2.5-coder:1.5b",
                "prompt": prompt,
                "stream": False
            }
            response = requests.post(ollama_url, json=payload).json()
            generated_code = response.get("response", "")

            
            f.write(json.dumps({
                "task_id": task_id,
                "solution": generated_code
            }) + "\n")

        monitoring = False
        thread.join()
        total_time = time.time()-start_time
        print(f"time: {total_time}\nenergy: {energy}\npeak_vram: {peak_vram}\n")
        pynvml.nvmlShutdown()

In [ ]:
humaneval_problems = get_human_eval_plus()
print(f"number of problems: {len(humaneval_problems)}")
generate_solutions(humaneval_problems, f"{path}/{humaneval_filename}")

number of problems: 164
NVIDIA GeForce RTX 4060
time: 484.58866333961487
energy: 0.0
peak_vram: 2234.08984375



In [ ]:
mbpp_problems = get_mbpp_plus()
print(f"number of problems: {len(mbpp_problems)}")
generate_solutions(mbpp_problems, f"{path}/{mbpp_filename}")

number of problems: 378
NVIDIA GeForce RTX 4060
time: 627.3944425582886
energy: 0.0
peak_vram: 2288.5859375



In [9]:
def clean_file(file):
    import re

    regex = r"```python\n(.*?)\n```"

    outputsamples = []
    with open(file, "r") as f:
        for line in f:
            sample = json.loads(line)
            solution = sample['solution']
            match = re.search(regex, solution, re.DOTALL)
            solution = match.group(1) if match else solution
            sample['solution'] = solution
            outputsamples.append(sample)


    with open(file, "w") as f:
        for sample in outputsamples:
            f.write(json.dumps(sample)+"\n")

In [18]:
clean_file(f"{path}humaneval_solutions.jsonl")
clean_file(f"{path}mbpp_solutions.jsonl")

In [20]:
!evalplus.evaluate --dataset humaneval --samples ../data/processed/qwen_15b/pass@1/humaneval_solutions.jsonl

Load from previous results from ../data/processed/qwen_15b/pass@1/humaneval_solutions_eval_results.json
humaneval (base tests)
pass@1:	0.646
humaneval+ (base + extra tests)
pass@1:	0.591


In [22]:
!evalplus.evaluate --dataset mbpp --samples ../data/processed/qwen_15b/pass@1/mbpp_solutions.jsonl

100% [........................................................] 336032 / 336032Computing expected output...
Expected outputs computed in 45.75s
Reading samples...
378it [00:00, 2082.89it/s]
100%|████████████████████████████████████████| 378/378 [00:31<00:00, 11.99it/s]
mbpp (base tests)
pass@1:	0.608
mbpp+ (base + extra tests)
pass@1:	0.511
